# AvisSense - Fine-tuning CamemBERT pour analyse de sentiment
## Classification d'avis français (positif / négatif)

### Installation (à faire une fois)

In [ ]:
# deja installe normalement, mais au cas ou
# %pip install torch transformers datasets scikit-learn pandas numpy

### 1. Imports

In [1]:
import warnings; warnings.filterwarnings("ignore")
import torch
import numpy as np
from transformers import CamembertTokenizer, CamembertForSequenceClassification
from transformers import Trainer, TrainingArguments
from datasets import load_dataset, DatasetDict
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device = {device}")

device = cpu


### 2. Charger le dataset Allociné

In [2]:
# on charge les avis cinema francais
dataset = load_dataset("tblard/allocine")
print(dataset)
print("\nExemple:", dataset["train"][0])

DatasetDict({
    train: Dataset({
        features: ['review', 'label'],
        num_rows: 160000
    })
    validation: Dataset({
        features: ['review', 'label'],
        num_rows: 20000
    })
    test: Dataset({
        features: ['review', 'label'],
        num_rows: 20000
    })
})

Exemple: {'review': 'Si vous cherchez du cinéma abrutissant à tous les étages,n\'ayant aucune peur du cliché en castagnettes et moralement douteux,"From Paris with love" est fait pour vous.Toutes les productions Besson,via sa filière EuropaCorp ont de quoi faire naître la moquerie.Paris y est encore une fois montrée comme une capitale exotique,mais attention si l\'on se dirige vers la banlieue,on y trouve tout plein d\'intégristes musulmans prêts à faire sauter le caisson d\'une ambassadrice américaine.Nauséeux.Alors on se dit qu\'on va au moins pouvoir apprécier la déconnade d\'un classique buddy-movie avec le jeune agent aux dents longues obligé de faire équipe avec un vieux lou complètement t

In [3]:
# regarder l'equilibre des classes
train_labels = dataset["train"]["label"]
n_pos = sum(train_labels)
n_neg = len(train_labels) - n_pos
print(f"Positifs: {n_pos}, Negatifs: {n_neg}, ratio: {n_pos/len(train_labels):.2f}")

Positifs: 80587, Negatifs: 79413, ratio: 0.50


### 3. Prendre un sous-ensemble (pour que ça tienne sur CPU)

In [ ]:
# le dataset complet c'est 200k avis, trop long sur CPU
# on prend 5000 train, 1000 val, 1000 test
train_small = dataset["train"].shuffle(seed=42).select(range(5000))
val_small = dataset["validation"].shuffle(seed=42).select(range(1000))
test_small = dataset["test"].shuffle(seed=42).select(range(1000))

small = DatasetDict({
    "train": train_small,
    "validation": val_small,
    "test": test_small
})
print(small)

# juste pour tester si le code fonctionne (100 avis, 1 epoch)
# train_small = dataset["train"].shuffle(seed=42).select(range(100))
# val_small = dataset["validation"].shuffle(seed=42).select(range(50))
# test_small = dataset["test"].shuffle(seed=42).select(range(50))

# small = DatasetDict({
#     "train": train_small,
#     "validation": val_small,
#     "test": test_small
# })
# print(small)

DatasetDict({
    train: Dataset({
        features: ['review', 'label'],
        num_rows: 100
    })
    validation: Dataset({
        features: ['review', 'label'],
        num_rows: 50
    })
    test: Dataset({
        features: ['review', 'label'],
        num_rows: 50
    })
})


### 4. Tokenization avec CamemBERT

In [5]:
# CamemBERT = BERT mais pour le francais (cree par Inria/Orange)
tokenizer = CamembertTokenizer.from_pretrained("camembert-base")

def tokenize_fn(examples):
    return tokenizer(examples["review"], truncation=True, padding="max_length", max_length=256)

# on tokenize tout le petit dataset
small_tokenized = small.map(tokenize_fn, batched=True)
print("tokenization terminee")
print("shape d'un exemple:", small_tokenized["train"][0]["input_ids"][:10])

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

tokenization terminee
shape d'un exemple: [5, 8518, 7997, 1813, 31, 33, 5708, 29, 838, 1610]


### 5. Charger CamemBERT pré-entraîné

In [6]:
# 2 labels : 0 = negatif, 1 = positif
model = CamembertForSequenceClassification.from_pretrained("camembert-base", num_labels=2)
model.to(device)
print(f"modele charge sur {device}")
print(f"nombre de parametres: {sum(p.numel() for p in model.parameters()):,}")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] CamembertForSequenceClassification LOAD REPORT from: camembert-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.bias                | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


modele charge sur cpu
nombre de parametres: 110,623,490


### 6. Définir la métrique d'évaluation

In [7]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds),
        "precision": precision_score(labels, preds),
        "recall": recall_score(labels, preds),
    }

### 7. Configurer et lancer l'entraînement

In [ ]:
# 3 epochs suffisent pour le fine-tuning (camemBERT deja entraine sur bcp de texte)
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    # num_train_epochs=1,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_tokenized["train"],
    eval_dataset=small_tokenized["validation"],
    compute_metrics=compute_metrics,
)

In [9]:
# on lance l entrainement (sur CPU ca prend ~5 minutes par epoch)
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,No log,0.693230,0.480000,0.648649,0.480000,1.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=13, training_loss=0.6914553275475135, metrics={'train_runtime': 63.0353, 'train_samples_per_second': 1.586, 'train_steps_per_second': 0.206, 'total_flos': 13155552768000.0, 'train_loss': 0.6914553275475135, 'epoch': 1.0})

### 8. Évaluation finale sur le test

In [10]:
results = trainer.evaluate(small_tokenized["test"])
print("\nResultats sur le test:")
for k, v in results.items():
    print(f"  {k}: {v:.4f}")

Training Loss,Validation Loss,Epoch,Accuracy,F1,Precision,Recall
No log,0.713063,1,0.440000,0.611111,0.440000,1.000000



Resultats sur le test:
  eval_loss: 0.7131
  eval_accuracy: 0.4400
  eval_f1: 0.6111
  eval_precision: 0.4400
  eval_recall: 1.0000


### 9. Sauvegarder le modèle

In [11]:
# on sauvegarde pour l utiliser dans app.py apres
model.save_pretrained("../model/saved_model")
tokenizer.save_pretrained("../model/saved_model")
print("modele sauvegarde dans model/saved_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

modele sauvegarde dans model/saved_model


### 10. Tester avec des avis perso

In [12]:
def predict(text):
    model.eval()
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=256)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=1)
        pred = torch.argmax(probs, dim=1).item()
    label = "POSITIF" if pred == 1 else "NEGATIF"
    confiance = probs[0][pred].item()
    return label, confiance

tests = [
    "Ce film est absolument magnifique, j'ai adore chaque minute.",
    "Quelle daube, j'ai jamais vu un film aussi nul de toute ma vie.",
    "Franchement c'est moyen, ca passe mais sans plus.",
    "Les acteurs jouent super bien mais le scenario est trop lent.",
]

for t in tests:
    label, conf = predict(t)
    print(f"{label} ({conf:.1%}) -> {t}")

POSITIF (57.5%) -> Ce film est absolument magnifique, j'ai adore chaque minute.
POSITIF (55.5%) -> Quelle daube, j'ai jamais vu un film aussi nul de toute ma vie.
POSITIF (54.6%) -> Franchement c'est moyen, ca passe mais sans plus.
POSITIF (55.2%) -> Les acteurs jouent super bien mais le scenario est trop lent.
